In [1]:
{
  "nbformat": 4,
  "nbformat_minor": 0,
  "metadata": {
    "colab": {
      "provenance": [],
      "authorship_tag": "ABX9TyP/i9Kjwsk+z9gS7mhJ4NZ8",
      "include_colab_link": True  # <--- Changed 'true' to 'True'
    },
    "kernelspec": {
      "name": "python3",
      "display_name": "Python 3"
    },
    "language_info": {
      "name": "python"
    }
  },
  "cells": [{
  "nbformat": 4,
  "nbformat_minor": 0,
  "metadata": {
    "colab": {
      "provenance": [],
      "authorship_tag": "ABX9TyP/i9Kjwsk+z9gS7mhJ4NZ8",
      "include_colab_link": true
    },
    "kernelspec": {
      "name": "python3",
      "display_name": "Python 3"
    },
    "language_info": {
      "name": "python"
    }
  },
  "cells": [
    {
      "cell_type": "markdown",
      "metadata": {
        "id": "view-in-github",
        "colab_type": "text"
      },
      "source": [
        "<a href=\"https://colab.research.google.com/github/monusait/Heavy-Equipment-Dashboard/blob/main/Streamlit_app.ipynb\" target=\"_parent\"><img src=\"https://colab.research.google.com/assets/colab-badge.svg\" alt=\"Open In Colab\"/></a>"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {
        "id": "d0d0779f"
      },
      "source": [
        "## Write Streamlit Application Script\n",
        "\n",
        "### Subtask:\n",
        "Create the `app.py` script containing the logic for the heavy equipment inspection system, including checklists, data logging, and Plotly visualizations.\n"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {
        "id": "9b51056e"
      },
      "source": [
        "**Reasoning**:\n",
        "I will create the `app.py` script for the Streamlit application using the `%%writefile` magic command. This script will include the inspection checklists, form logic for data entry, persistent CSV logging, and an analytics dashboard with interactive Plotly visualizations as specified in the subtask instructions.\n",
        "\n"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {
        "colab": {
          "base_uri": "https://localhost:8080/"
        },
        "id": "a530921e",
        "outputId": "7ce95b0d-ef87-4d8a-fda9-a42bad31ba8c"
      },
      "source": [
        "%%writefile app.py\n",
        "import streamlit as st\n",
        "import pandas as pd\n",
        "import os\n",
        "from datetime import date\n",
        "import plotly.express as px\n",
        "\n",
        "# 1. Define inspection checklists\n",
        "inspection_checklists = {\n",
        "    'Forklift': [\n",
        "        'Horn operation', 'Steering responsiveness', 'Braking system (service and parking)',\n",
        "        'Fluid levels (oil, fuel, coolant, hydraulic)', 'Fork condition (cracks, bends, wear)',\n",
        "        'Mast and chain condition', 'Tire condition and pressure', 'Lights and alarms'\n",
        "    ],\n",
        "    'Crane': [\n",
        "        'Wire rope condition (broken wires, kinking, wear)', 'Hook latches and condition',\n",
        "        'Limit switches operation', 'Load testing indicators and load charts',\n",
        "        'Hydraulic or mechanical control function', 'Outriggers or stabilizers',\n",
        "        'Drum and sheave condition'\n",
        "    ],\n",
        "    'Shackle': [\n",
        "        'Pin wear and condition', 'Body deformation (twisting or stretching)',\n",
        "        'Legible markings (WLL and manufacturer ID)', 'Cracks or nicks',\n",
        "        'Proper pin seating and engagement'\n",
        "    ]\n",
        "}\n",
        "\n",
        "LOG_FILE = 'heavy_equipment_inspections.csv'\n",
        "\n",
        "# 2. App Page Config\n",
        "st.set_page_config(page_title=\"Heavy Equipment Inspection System\", layout=\"wide\")\n",
        "st.title(\"⚙️ Heavy Equipment Inspection System\")\n",
        "\n",
        "# 3. Streamlit Navigation Tabs\n",
        "tab1, tab2 = st.tabs([\"Daily Inspection Form\", \"Analytics Dashboard\"])\n",
        "\n",
        "with tab1:\n",
        "    st.header(\"Safety Check Entry\")\n",
        "\n",
        "    # Layout for metadata\n",
        "    col_meta1, col_meta2 = st.columns(2)\n",
        "    with col_meta1:\n",
        "        equipment_type = st.selectbox(\"Equipment Type\", list(inspection_checklists.keys()))\n",
        "        inspector = st.text_input(\"Inspector Name\")\n",
        "    with col_meta2:\n",
        "        equip_id = st.text_input(\"Equipment ID\")\n",
        "        entry_date = st.date_input(\"Date\", value=date.today())\n",
        "\n",
        "    st.divider()\n",
        "    st.subheader(f\"{equipment_type} Checklist Items\")\n",
        "\n",
        "    # Checklist Logic\n",
        "    items = inspection_checklists[equipment_type]\n",
        "    responses = {}\n",
        "\n",
        "    # Organize checklist into two columns\n",
        "    col_chk1, col_chk2 = st.columns(2)\n",
        "    for i, item in enumerate(items):\n",
        "        current_col = col_chk1 if i % 2 == 0 else col_chk2\n",
        "        responses[item] = current_col.selectbox(item, options=['Pass', 'Fail', 'N/A'], key=f\"{equipment_type}_{item}\")\n",
        "\n",
        "    if st.button(\"Submit Inspection\", type=\"primary\"):\n",
        "        if not inspector or not equip_id:\n",
        "            st.error(\"Inspector Name and Equipment ID are required.\")\n",
        "        else:\n",
        "            # Prepare Record\n",
        "            new_record = {\n",
        "                'Date': str(entry_date),\n",
        "                'Inspector': inspector,\n",
        "                'Equipment_Type': equipment_type,\n",
        "                'Equipment_ID': equip_id\n",
        "            }\n",
        "            new_record.update(responses)\n",
        "            df_new = pd.DataFrame([new_record])\n",
        "\n",
        "            # Log to CSV\n",
        "            if not os.path.isfile(LOG_FILE):\n",
        "                df_new.to_csv(LOG_FILE, index=False)\n",
        "            else:\n",
        "                # Ensure we handle columns correctly by loading existing first or appending carefully\n",
        "                df_new.to_csv(LOG_FILE, mode='a', header=False, index=False)\n",
        "\n",
        "            st.success(\"Inspection logged successfully!\")\n",
        "\n",
        "            # Maintenance Alerts\n",
        "            failed_items = [k for k, v in responses.items() if v == 'Fail']\n",
        "            if failed_items:\n",
        "                st.warning(f\"⚠️ MAINTENANCE ALERT: Critical items failed: {', '.join(failed_items)}\")\n",
        "            else:\n",
        "                st.info(\"✅ Equipment passed all safety checks.\")\n",
        "\n",
        "with tab2:\n",
        "    st.header(\"Performance & Compliance Analytics\")\n",
        "\n",
        "    if not os.path.exists(LOG_FILE):\n",
        "        st.info(\"No data logged yet. Please submit an inspection record to view analytics.\")\n",
        "    else:\n",
        "        df = pd.read_csv(LOG_FILE)\n",
        "        df['Date'] = pd.to_datetime(df['Date'])\n",
        "\n",
        "        # Visualization 1: Inspection Frequency\n",
        "        st.subheader(\"Inspection Volume Trends\")\n",
        "        trend_df = df.groupby(df['Date'].dt.date).size().reset_index(name='Daily Inspections')\n",
        "        fig_line = px.line(trend_df, x='Date', y='Daily Inspections', markers=True,\n",
        "                          title=\"Daily Inspection Count\", labels={'Date': 'Inspection Date'})\n",
        "        st.plotly_chart(fig_line, use_container_width=True)\n",
        "\n",
        "        # Visualization 2: Equipment Health Status\n",
        "        st.subheader(\"Health Status Overview\")\n",
        "        meta_cols = ['Date', 'Inspector', 'Equipment_Type', 'Equipment_ID']\n",
        "        checklist_cols = [c for c in df.columns if c not in meta_cols]\n",
        "\n",
        "        # Reshape for categorical plotting\n",
        "        melted_df = df.melt(id_vars=['Equipment_Type'], value_vars=checklist_cols, value_name='Status')\n",
        "        fig_bar = px.histogram(melted_df, x='Equipment_Type', color='Status', barmode='group',\n",
        "                              title=\"Status Counts by Equipment Type\",\n",
        "                              color_discrete_map={'Pass': 'green', 'Fail': 'red', 'N/A': 'gray'})\n",
        "        st.plotly_chart(fig_bar, use_container_width=True)\n",
        "\n",
        "print('Streamlit app.py has been created.')"
      ],
      "execution_count": 38,
      "outputs": [
        {
          "output_type": "stream",
          "name": "stdout",
          "text": [
            "Overwriting app.py\n"
          ]
        }
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {
        "id": "9f12c048"
      },
      "source": [
        "## Install Streamlit\n",
        "\n",
        "### Subtask:\n",
        "Ensure the 'streamlit' library is installed in the current environment.\n"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {
        "id": "3ba88567"
      },
      "source": [
        "**Reasoning**:\n",
        "Install the Streamlit library quietly to prepare the environment for the web application development.\n",
        "\n"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {
        "colab": {
          "base_uri": "https://localhost:8080/"
        },
        "id": "68928cb2",
        "outputId": "ffc96201-b111-488e-8176-e7c4bee9ca9d"
      },
      "source": [
        "!pip install streamlit -q\n",
        "print(\"Streamlit installed successfully.\")"
      ],
      "execution_count": 39,
      "outputs": [
        {
          "output_type": "stream",
          "name": "stdout",
          "text": [
            "\u001b[2K   \u001b[90m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[32m9.1/9.1 MB\u001b[0m \u001b[31m44.3 MB/s\u001b[0m eta \u001b[36m0:00:00\u001b[0m\n",
            "\u001b[2K   \u001b[90m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[32m6.9/6.9 MB\u001b[0m \u001b[31m93.3 MB/s\u001b[0m eta \u001b[36m0:00:00\u001b[0m\n",
            "\u001b[?25hStreamlit installed successfully.\n"
          ]
        }
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {
        "id": "e3d2e334"
      },
      "source": [
        "**Reasoning**:\n",
        "The 'streamlit' library has been successfully installed in the environment. The next step is to deploy the Streamlit application in a background process and generate a secure proxy link for access within the Google Colab environment.\n",
        "\n"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {
        "colab": {
          "base_uri": "https://localhost:8080/",
          "height": 72
        },
        "id": "17f4e6d8",
        "outputId": "7b21baea-2fbf-4a80-8d45-814900066e8b"
      },
      "source": [
        "import subprocess\n",
        "import time\n",
        "from google.colab import output\n",
        "\n",
        "# 1. Define the port for the Streamlit server\n",
        "STREAMLIT_PORT = 8501\n",
        "\n",
        "# 2. Launch the Streamlit app in the background using subprocess\n",
        "# --server.port ensures it runs on the expected port\n",
        "# --server.headless=true is required for cloud/headless environments\n",
        "process = subprocess.Popen(['streamlit', 'run', 'app.py', '--server.port', str(STREAMLIT_PORT), '--server.headless', 'true'])\n",
        "\n",
        "# Give the server a few seconds to initialize\n",
        "time.sleep(5)\n",
        "\n",
        "# 3. Generate the public proxy URL using Colab's kernel utility\n",
        "proxy_url = output.eval_js(f\"google.colab.kernel.proxyPort({STREAMLIT_PORT})\")\n",
        "\n",
        "print(f\"Streamlit server is running in the background (PID: {process.pid}).\")\n",
        "print(f\"Access your Heavy Equipment Inspection System here: {proxy_url}\")"
      ],
      "execution_count": 40,
      "outputs": [
        {
          "output_type": "stream",
          "name": "stdout",
          "text": [
            "Streamlit server is running in the background (PID: 16224).\n",
            "Access your Heavy Equipment Inspection System here: https://8501-m-s-31i6n2ps1pqo9-c.us-central1-0.prod.colab.dev\n"
          ]
        }
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {
        "id": "fd52dddf"
      },
      "source": [
        "## Final Task\n",
        "\n",
        "### Subtask:\n",
        "Summarize the Streamlit application features and provide the final access link.\n"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {
        "id": "a2dc0d85"
      },
      "source": [
        "## Summary:\n",
        "\n",
        "### Q&A\n",
        "\n",
        "**What are the primary features of the Heavy Equipment Inspection System?**\n",
        "The system features specialized safety checklists for forklifts, cranes, and shackles based on Alberta OHS requirements. It includes a digital entry form for inspectors, a persistent CSV-based logging mechanism, automated maintenance alerts for failed components, and an interactive Plotly dashboard for monitoring compliance and equipment health trends.\n",
        "\n",
        "**Where can the application be accessed?**\n",
        "The application is hosted via a secure Google Colab kernel proxy at the following URL: [https://8501-m-s-31i6n2ps1pqo9-c.us-central1-0.prod.colab.dev](https://8501-m-s-31i6n2ps1pqo9-c.us-central1-0.prod.colab.dev).\n",
        "\n",
        "### Data Analysis Key Findings\n",
        "\n",
        "*   **Comprehensive Safety Coverage:** The application incorporates 20 distinct safety parameters across three equipment categories: 8 for Forklifts (e.g., steering, forks, mast), 7 for Cranes (e.g., wire rope, limit switches), and 5 for Shackles (e.g., pin wear, markings).\n",
        "*   **Dynamic Maintenance Alerts:** The system identifies critical failures in real-time. If an inspector selects \"Fail\" for any checklist item, the app generates a specific warning message listing the failed components to prevent the operation of unsafe equipment.\n",
        "*   **Data Persistence and Structure:** All inspections are captured in a structured format within heavy\\_equipment\\_inspections.csv, recording the date, inspector name, equipment ID, and the status of every safety check.\n",
        "*   **Visual Compliance Monitoring:** The dashboard provides high-level oversight through two primary visualizations: a line chart tracking daily inspection volume and a grouped histogram showing the distribution of \"Pass,\" \"Fail,\" and \"N/A\" statuses by equipment type.\n",
        "\n",
        "### Insights or Next Steps\n",
        "\n",
        "*   **Automated Notifications:** Integrate a notification system (such as email or SMS alerts) that automatically flags safety managers the moment a \"Fail\" status is submitted, ensuring immediate lockout/tagout procedures.\n",
        "*   **Cloud Data Integration:** Transition the local CSV storage to a cloud-based database (e.g., BigQuery or Google Sheets API) to enable long-term data durability and allow multiple inspectors to sync data simultaneously from different locations.\n"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {
        "id": "2b17ee16"
      },
      "source": [],
      "execution_count": 43,
      "outputs": []
    },
    {
      "cell_type": "code",
      "metadata": {
        "id": "e8d69c1d"
      },
      "source": [],
      "execution_count": 43,
      "outputs": []
    },
    {
      "cell_type": "code",
      "metadata": {
        "id": "19aa1de4"
      },
      "source": [],
      "execution_count": 43,
      "outputs": []
    }
  ]
}

_IncompleteInputError: incomplete input (1444978730.py, line 384)